# GPT-2 Inference Engine - Google Colab Setup

This notebook sets up the GPT-2 inference engine on Google Colab with T4 GPU acceleration.

**Runtime:** GPU (T4) | **Estimated time:** 15-20 minutes

## Step 1: Environment Setup

Check GPU availability and install build dependencies.

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install build dependencies
!apt-get update && apt-get install -y build-essential cmake git wget curl htop

In [ ]:
# Check CUDA version
!nvcc --version

## Step 2: Clone and Build GGML with CUDA Support

GGML provides the tensor computation backend with CUDA acceleration.

In [ ]:
# Clone GGML
%cd /content
!git clone https://github.com/ggerganov/ggml.git

In [ ]:
# Build GGML with CUDA (using CUBLAS backend - more stable in Colab)
%cd /content/ggml
!mkdir -p build && cd build && cmake .. -DCMAKE_CUDA_PATH=/usr/local/cuda -DGGML_CUBLAS=ON -DCMAKE_BUILD_TYPE=Release && make -j$(nproc)

In [ ]:
# Verify GGML build
%cd /content/ggml/build && ls -la lib/

## Step 3: Clone Inference Engine

Copy your inference engine source files to the Colab environment.

In [ ]:
# Option A: If using Google Drive (mount first)
# from google.colab import drive
# drive.mount('/content/gdrive')

# Option B: If uploading files directly - use the Files panel (left sidebar)
# Or clone from GitHub if hosted:
# !git clone https://github.com/YOUR_USERNAME/inference-engine.git

# For now, we'll create the project structure (only if not already cloned)
# Note: git clone in next cell will handle directory creation

In [ ]:
# Clone the inference engine repo (includes CMakeLists.txt)
%cd /content
!rm -rf inference-engine && git clone https://github.com/nisbenz/inference-engine.git inference-engine

# If your repo is private or not yet pushed, manually create structure:
# !mkdir -p /content/inference-engine/src

# Verify CMakeLists.txt exists
!ls -la /content/inference-engine/

In [ ]:
# Upload your source files
# Use the Files panel to upload src/model.cpp, src/model.hpp, src/layers.cpp, 
# src/layers.hpp, src/tokenizer.cpp, src/tokenizer.hpp, src/kv_cache.cpp, 
# src/kv_cache.hpp, src/main.cpp

# Or run this cell after uploading via the Files panel
import os

src_dir = '/content/inference-engine/src'
print(f'Source files in {src_dir}:')
for f in os.listdir(src_dir):
    print(f'  {f}')

## Step 4: Download GPT-2 Model (GGUF Format)

Download pre-converted GPT-2 GGUF model directly.

In [ ]:
# Install HuggingFace Hub for downloading models
!pip install huggingface_hub -q

In [ ]:
# Download GPT-2 GGUF directly from HuggingFace
# Using QuantFactory's GPT-2 GGUF model

%cd /content
!pip install huggingface_hub -q

from huggingface_hub import hf_hub_download
from huggingface_hub import list_repo_files

# List available GGUF files
print("Available files in QuantFactory/gpt2-GGUF:")
files = list_repo_files("QuantFactory/gpt2-GGUF")
for f in files:
    if f.endswith('.gguf'):
        print(f"  {f}")

In [ ]:
# Download GPT-2 GGUF model (adjust filename based on available files above)
# Using Q4_K_M quantization for balance of speed and quality

print("Downloading GPT-2 GGUF model...")
model_path = hf_hub_download(
    repo_id="QuantFactory/gpt2-GGUF",
    filename="gpt2.Q4_K_M.gguf",
    repo_type="model",
    local_dir="/content/gpt2-model"
)
print(f"Model downloaded to: {model_path}")

In [ ]:
# Verify GGUF file
!ls -lh /content/gpt2-model/

In [ ]:
# Verify GGUF file
!ls -lh /content/gpt2-model/

## Step 5: Update Source Code for GGUF Loading

The current `load_weights()` needs to be updated to parse GGUF format.

In [ ]:
# Create updated model.cpp with GGUF loading support
# This replaces load_weights() with actual GGUF implementation

model_cpp_content = '''
// ... (rest of your existing model.cpp)

bool GPT2Model::load_weights(const std::string& path) {
    // Try GGUF format first
    struct gguf_context* ctx = gguf_load(path.c_str());
    if (ctx) {
        printf("Loaded GGUF model from: %s\\n", path.c_str());
        
        // Get tensor count
        int n_tensors = gguf_get_n_tensors(ctx);
        printf("Number of tensors: %d\\n", n_tensors);
        
        // Load each tensor
        for (int i = 0; i < n_tensors; i++) {
            const char* name = gguf_get_tensor_name(ctx, i);
            struct ggml_tensor* tensor = ggml_get_tensor(ctx_, name);
            if (tensor) {
                // Read tensor data from file
                // This is handled by ggml_init_from_file
            }
        }
        
        gguf_free(ctx);
        return true;
    }
    
    // Fallback to binary format
    return load_ggml_weights(path);
}
'''

print('GGUF loading code template created')
print('Note: You need to integrate this with your actual model.cpp')

In [ ]:
# (Build step moved to Step 6)

In [ ]:
# (Inference run moved to Step 7)

## Step 6: Build Custom Inference Engine

Build your own inference engine with the updated source code.

In [ ]:
# Build the inference engine
%cd /content/inference-engine
!mkdir -p build && cd build && cmake .. -DCMAKE_CUDA_PATH=/usr/local/cuda -DCMAKE_CUDA_ARCHITECTURES=75 && make -j$(nproc)

In [ ]:
# Verify build
%cd /content/inference-engine/build && ls -la

## Step 7: Run Inference

Execute your inference engine.

In [ ]:
# Run with your inference engine
%cd /content/inference-engine/build && ./gpt2 "Hello, world!" 50 0.8 40

## Troubleshooting

### Common Issues:

1. **CUDA out of memory**: Reduce batch size or use smaller quantization (e.g., Q3_K_M instead of Q4_K_M)
2. **GGUF download fails**: Check HuggingFace connection, try alternative model repo
3. **Build errors**: Ensure all source files are uploaded to `/content/inference-engine/src/`
4. **Tensor mismatch**: GPT-2 Large (774M) config: 36 layers, 20 heads, 1280 hidden size

In [ ]:
# Check GGUF model metadata
!pip install numpy sentencepiece -q
import struct

def read_gguf_header(path):
    with open(path, 'rb') as f:
        magic = struct.unpack('I', f.read(4))[0]
        print(f'Magic: {hex(magic)}')
        version = struct.unpack('I', f.read(4))[0]
        print(f'Version: {version}')

import glob
gguf_files = glob.glob('/content/gpt2-model/*.gguf')
if gguf_files:
    read_gguf_header(gguf_files[0])
else:
    print("No GGUF file found")

## File Summary

After uploading your source files, the structure should be:
```
/content/
├── ggml/                    # GGML library (CUDA enabled)
├── inference-engine/        # Your project
│   ├── src/
│   │   ├── model.cpp
│   │   ├── model.hpp
│   │   ├── layers.cpp
│   │   ├── layers.hpp
│   │   ├── tokenizer.cpp
│   │   ├── tokenizer.hpp
│   │   ├── kv_cache.cpp
│   │   └── kv_cache.hpp
│   ├── build/
│   └── CMakeLists.txt
└── gpt2-model/              # GPT-2 GGUF model file (~500MB for Q4_K_M)
```

## Next Steps

1. Upload your source files to `/content/inference-engine/src/`
2. Update `model.cpp` to implement GGUF loading
3. Run cells from Step 6 onwards
4. For custom inference logic, modify the model classes